# VBF HH bbtautau SPANet truth validation

Set `BASE` to the directory containing the reduced parquet files. The cells below validate the LHEPart -> GenJet -> SPANetJet VBF labels and GenPart Higgs -> b b labels before training.

In [ ]:
from pathlib import Path
import sys

SPANET = Path('/local/norman/UHHAnalysis/SPANet')
sys.path.insert(0, str(SPANET / 'utils'))

from convert_hh_bbtautau_parquet import build_dataset, expand_inputs, load_chunks
from validate_hh_bbtautau_truth import summarize, print_event_dump

BASE = Path('/local/norman/UHHAnalysis/SPANet/22pre_v14')
SAMPLE = BASE / 'hh_vbf_hbb_htt_kv1_k2v1_kl1_madgraph/nominal/calib__default/sel__default/red__default/prod24'
paths = sorted(SAMPLE.glob('events_*.parquet'))
paths[:3], len(paths)

In [ ]:
chunks = load_chunks(paths[:1], max_events=5000)
dataset = build_dataset(
    chunks,
    max_jets=None,
    source_collection='SPANetJet',
    hhb_dr=0.4,
    vbf_truth='genjet',
    lhe_to_gen_dr=0.4,
    gen_to_reco_dr=0.3,
    use_overlap_cleaning=True,
)
summarize(dataset)
print_event_dump(chunks, dataset, n_events=5, source_collection='SPANetJet')

In [ ]:
# Convert all current 22pre_v14 VBF samples into one HDF5 after the validation above looks reasonable.
!python /local/norman/UHHAnalysis/SPANet/utils/convert_hh_bbtautau_parquet.py \
  /local/norman/UHHAnalysis/SPANet/22pre_v14/hh_vbf_hbb_htt_*_madgraph \
  -o /local/norman/UHHAnalysis/SPANet/data/vbf_hh_bbtautau/22pre_v14_vbf_signal.h5 \
  --source-collection SPANetJet \
  --vbf-truth genjet

In [ ]:
# Tiny CPU/GPU smoke test. Use this only after the HDF5 was created.
!cd /local/norman/UHHAnalysis/SPANet && python -m spanet.train \
  -ef event_files/vbf_hh_bbtautau.yaml \
  -tf data/vbf_hh_bbtautau/22pre_v14_vbf_signal.h5 \
  -of options_files/vbf_hh_bbtautau/smoke.json \
  -l outputs \
  -n vbf_hh_bbtautau_smoke \
  -e 1 \
  -b 64 \
  -p 5